In [ ]:
import sys
from pathlib import Path

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

In [ ]:
project_root = Path.cwd().parent
raw_data_dir = project_root / "data" / "raw"

raw_data_dir.mkdir(parents=True, exist_ok=True)

print("Raw document folder:", raw_data_dir)
print("Folder exists:", raw_data_dir.is_dir())

In [ ]:
for file_path in sorted(raw_data_dir.iterdir()):
    if file_path.is_file():
        print(file_path.name, "|", file_path.stat().st_size, "bytes")

In [ ]:
from docx import Document

report_path = raw_data_dir / "microsoft_2024_annual_report.docx"
report = Document(report_path)

nonempty_paragraphs = [
    paragraph for paragraph in report.paragraphs
    if paragraph.text.strip()
]

print("Nonempty paragraphs:", len(nonempty_paragraphs))
print("Tables:", len(report.tables))

for paragraph in nonempty_paragraphs[:10]:
    print(f"\n[{paragraph.style.name}] {paragraph.text[:400]}")

In [ ]:
table = report.tables[0]

print("Rows:", len(table.rows))
print("Columns:", len(table.columns))

for row_number, row in enumerate(table.rows[:8], start=1):
    cells = [cell.text for cell in row.cells]
    print(f"Row {row_number}: {cells}")

In [ ]:
def normalize_whitespace(text):
    return " ".join(text.split())

cleaned_rows = [
    [normalize_whitespace(cell.text) for cell in row.cells]
    for row in table.rows
]

for row_number, cells in enumerate(cleaned_rows, start=1):
    if any(cells):
        print(f"Row {row_number}: {cells}")

In [ ]:
from docx.text.paragraph import Paragraph
from docx.table import Table

preceding_paragraphs = []

for block in report.iter_inner_content():
    if isinstance(block, Paragraph):
        text = normalize_whitespace(block.text)
        if text:
            preceding_paragraphs.append(text)
    elif isinstance(block, Table):
        break

for text in preceding_paragraphs[-5:]:
    print(text, "\n")

In [ ]:
document_metadata = {
    "document_id": "microsoft_annual_report_2024",
    "company": "Microsoft",
    "report_fiscal_year": 2024,
    "document_type": "annual_report",
    "source_file": report_path.name,
    "source_url": (
        "https://www.microsoft.com/investor/reports/ar24/download-center/"
    ),
    "file_format": report_path.suffix.lstrip("."),
}

document_metadata

In [ ]:
paragraph_records = []

for block_index, block in enumerate(report.iter_inner_content()):
    if isinstance(block, Paragraph):
        raw_text = block.text
        text = normalize_whitespace(raw_text)

        if text:
            paragraph_records.append({
                **document_metadata,
                "block_id": f"{document_metadata['document_id']}:block:{block_index}",
                "block_index": block_index,
                "block_type": "paragraph",
                "style": block.style.name,
                "raw_text": raw_text,
                "text": text,
            })

print("Paragraph records:", len(paragraph_records))
print("First block ID:", paragraph_records[0]["block_id"])
print("First text:", paragraph_records[0]["text"])

In [ ]:
table_records = []

for block_index, block in enumerate(report.iter_inner_content()):
    if isinstance(block, Table):
        raw_rows = [
            [cell.text for cell in row.cells]
            for row in block.rows
        ]

        table_records.append({
            **document_metadata,
            "block_id": f"{document_metadata['document_id']}:block:{block_index}",
            "block_index": block_index,
            "block_type": "table",
            "raw_rows": raw_rows,
            "rows": [
                [normalize_whitespace(cell) for cell in row]
                for row in raw_rows
            ],
        })

print("Table records:", len(table_records))
print("First table ID:", table_records[0]["block_id"])
print("First table header:", table_records[0]["rows"][1])

In [ ]:
document_records = sorted(
    paragraph_records + table_records,
    key=lambda record: record["block_index"],
)

first_table_position = next(
    i for i, record in enumerate(document_records)
    if record["block_type"] == "table"
)

print("Total records:", len(document_records))

for record in document_records[
    max(0, first_table_position - 2):first_table_position + 1
]:
    preview = (
        record["text"][:250]
        if record["block_type"] == "paragraph"
        else f"Table with {len(record['rows'])} rows"
    )
    print(record["block_index"], record["block_type"], preview)

In [ ]:
document_records = sorted(
    paragraph_records + table_records,
    key=lambda record: record["block_index"],
)

first_table_position = next(
    i for i, record in enumerate(document_records)
    if record["block_type"] == "table"
)

print("Total records:", len(document_records))

for record in document_records[
    max(0, first_table_position - 2):first_table_position + 1
]:
    preview = (
        record["text"][:250]
        if record["block_type"] == "paragraph"
        else f"Table with {len(record['rows'])} rows"
    )
    print(record["block_index"], record["block_type"], preview)

In [ ]:
block_ids = [record["block_id"] for record in document_records]
block_indices = [record["block_index"] for record in document_records]

assert len(block_ids) == len(set(block_ids)), "Duplicate block IDs found"
assert block_indices == sorted(block_indices), "Records are out of order"
assert all(
    record["document_id"] == document_metadata["document_id"]
    for record in document_records
), "Unexpected document ID"

print(f"Integrity checks passed for {len(document_records)} records.")

In [ ]:
import json

processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_data_dir / "microsoft_2024_records.jsonl"

with output_path.open("w", encoding="utf-8") as file:
    for record in document_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved records:", len(document_records))
print("File:", output_path)
print("Size:", output_path.stat().st_size, "bytes")

In [ ]:
with output_path.open("r", encoding="utf-8") as file:
    loaded_records = [json.loads(line) for line in file if line.strip()]

assert loaded_records == document_records, "Saved and loaded records differ"

print("Loaded records:", len(loaded_records))
print("Round-trip check passed.")